# 26 - Regresion lineal simple en R

## Proposito de la sesion

En esta sesion vas a aprender regresion lineal simple desde cero.

La idea no es solo correr `lm()`, sino entender:

1. que problema resuelve,
2. como se ve geometricamente,
3. que significan pendiente e intercepto,
4. como se ajusta una recta a los datos,
5. como interpretar predicciones y residuos,
6. que mide `R^2`,
7. y cuando una recta deja de ser una buena idea.


In [ ]:
# Configuracion minima y carga de datos.
options(scipen = 999)
data(cars)

head(cars)
summary(cars)


## Ruta de la sesion

Vamos a seguir este camino:

1. visualizar los datos,
2. plantear el modelo matematico,
3. calcular la recta manualmente,
4. ajustarla con `lm()`,
5. interpretar coeficientes,
6. revisar predicciones y residuos,
7. medir ajuste con `R^2`,
8. mirar diagnosticos y limites,
9. cerrar con ejercicios para pensar.

Trabajaremos con el dataset `cars`, donde:

- `speed` es la velocidad del auto,
- `dist` es la distancia de frenado.


In [ ]:
# Visualizacion inicial del problema.
plot(
  cars$speed,
  cars$dist,
  pch = 19,
  col = 'steelblue',
  xlab = 'speed',
  ylab = 'dist',
  main = 'Autos: velocidad vs distancia de frenado'
)


## 1) Que problema resuelve la regresion lineal simple

La regresion lineal simple intenta describir la relacion entre una variable explicativa `x` y una variable respuesta `y` usando una recta.

Aqui queremos responder una pregunta como esta:

> Si aumenta la velocidad, como cambia en promedio la distancia de frenado?

Eso es distinto de clasificar o agrupar.
Aqui la salida es numerica y continua.

Tambien es importante una advertencia: que dos variables esten relacionadas no significa automaticamente causalidad.
Una recta puede describir asociacion sin demostrar causa.


In [ ]:
# Resumen numerico rapido.
data.frame(
  variable = c('speed', 'dist'),
  media = round(c(mean(cars$speed), mean(cars$dist)), 3),
  desvio = round(c(sd(cars$speed), sd(cars$dist)), 3)
)

cor(cars$speed, cars$dist)


## 2) La ecuacion del modelo

La forma mas comun de escribir la regresion lineal simple es:

$$y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$$

donde:

- `\beta_0` es el **intercepto**,
- `\beta_1` es la **pendiente**,
- `\varepsilon_i` es el error o parte no explicada por la recta.

La prediccion promedio para un valor dado de `x` se escribe como:

$$\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 x$$

Interpretacion inicial:

- la pendiente dice cuanto cambia `y` cuando `x` aumenta una unidad,
- el intercepto es el valor esperado de `y` cuando `x = 0`.

Eso ultimo no siempre tiene sentido practico. A veces solo es una pieza algebraica de la recta.


In [ ]:
# Calculo manual de los coeficientes.
x <- cars$speed
y <- cars$dist

beta1_hat <- cov(x, y) / var(x)
beta0_hat <- mean(y) - beta1_hat * mean(x)

data.frame(
  coeficiente = c('intercepto', 'pendiente'),
  estimacion_manual = round(c(beta0_hat, beta1_hat), 6)
)


## 3) Que significa "ajustar" la recta

La recta no se dibuja a ojo.
Se elige para que los errores sean pequenos en conjunto.

Para cada observacion:

$$e_i = y_i - \hat{y}_i$$

A esos errores se les llama **residuos**.

El criterio clasico de minimos cuadrados busca minimizar:

$$\sum_{i=1}^{n} e_i^2 = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Por eso los errores grandes pesan mucho: se elevan al cuadrado.


In [ ]:
# Ajuste con la funcion de R para regresion lineal.
model <- lm(dist ~ speed, data = cars)

coef_lm <- coef(model)

data.frame(
  coeficiente = c('intercepto', 'pendiente'),
  manual = round(c(beta0_hat, beta1_hat), 6),
  lm = round(as.numeric(coef_lm), 6)
)


## 4) Ver la recta sobre los datos

Una vez ajustado el modelo, podemos dibujar la recta encima de la nube de puntos.

Eso ayuda a responder dos preguntas rapidas:

1. la tendencia general parece realmente lineal?
2. los puntos quedan muy dispersos alrededor de la recta o relativamente cerca?


In [ ]:
# Nube de puntos con la recta ajustada.
plot(
  cars$speed,
  cars$dist,
  pch = 19,
  col = 'steelblue',
  xlab = 'speed',
  ylab = 'dist',
  main = 'Recta de regresion ajustada'
)
abline(model, col = 'firebrick', lwd = 2)


## 5) Interpretacion de los coeficientes

La pendiente es la parte mas util del modelo.

Si la pendiente es positiva, valores mayores de `speed` se asocian con valores mayores de `dist`.
Si la pendiente es negativa, ocurre lo contrario.

En este problema, una pendiente positiva tiene sentido fisico: a mayor velocidad, mayor distancia de frenado.

El intercepto puede ser menos intuitivo.
Si `speed = 0` queda fuera del uso real del problema, no conviene sobreinterpretarlo.


In [ ]:
# Predicciones para algunas velocidades concretas.
new_speeds <- data.frame(speed = c(10, 15, 20, 25))
pred_simple <- predict(model, newdata = new_speeds)

data.frame(
  speed = new_speeds$speed,
  dist_predicha = round(pred_simple, 3)
)


## 6) Residuos: donde falla la recta en cada punto

Un residuo es la diferencia entre el valor observado y el valor predicho.

- residuo positivo: el punto quedo por encima de la recta,
- residuo negativo: el punto quedo por debajo,
- residuo cercano a cero: la recta paso cerca de ese punto.

Los residuos no son un detalle tecnico menor.
Son la forma mas directa de revisar que tanto deja sin explicar el modelo.


In [ ]:
# Tabla corta de observados, predichos y residuos.
res_df <- data.frame(
  speed = cars$speed,
  dist_observada = y,
  dist_predicha = fitted(model),
  residuo = resid(model)
)

head(round(res_df, 3), 10)

plot(
  fitted(model),
  resid(model),
  pch = 19,
  col = 'darkorange',
  xlab = 'valores ajustados',
  ylab = 'residuos',
  main = 'Residuos vs valores ajustados'
)
abline(h = 0, lty = 2, col = 'gray40')


## 7) Como medir el ajuste: SSE, RMSE y `R^2`

Tres cantidades utiles:

1. **SSE**: suma de cuadrados de los residuos.
2. **RMSE**: tamano tipico del error en unidades de la variable respuesta.
3. **R^2**: proporcion de variabilidad de `y` explicada por el modelo.

Las formulas basicas son:

$$SSE = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

$$R^2 = 1 - \frac{SSE}{SST}$$

con

$$SST = \sum_{i=1}^{n} (y_i - \bar{y})^2$$

`R^2` no mide causalidad ni garantiza que el modelo sea bueno para cualquier objetivo.
Solo resume cuanto de la variacion total queda capturada por la recta.


In [ ]:
# Calculo manual de metricas de ajuste.
y_hat <- fitted(model)
res <- resid(model)

sse <- sum(res^2)
sst <- sum((y - mean(y))^2)
rmse <- sqrt(mean(res^2))
r2_manual <- 1 - sse / sst
r2_lm <- summary(model)$r.squared

data.frame(
  metrica = c('SSE', 'RMSE', 'R2_manual', 'R2_lm'),
  valor = round(c(sse, rmse, r2_manual, r2_lm), 6)
)


## 8) Leer el resumen del modelo

`summary(model)` agrega varias piezas utiles:

- coeficientes estimados,
- error estandar de los coeficientes,
- valor t y p-value,
- error residual,
- `R^2` y `R^2` ajustado.

En una primera sesion no hace falta memorizar todos esos elementos, pero si conviene empezar a reconocerlos.


In [ ]:
# Resumen clasico del modelo lineal.
summary(model)


## 9) Intervalos: no es lo mismo estimar la media que predecir un caso nuevo

Dos ideas distintas:

- **intervalo de confianza**: rango plausible para la respuesta promedio en un valor de `x`.
- **intervalo de prediccion**: rango plausible para una observacion individual nueva.

El intervalo de prediccion suele ser mas ancho porque incluye tanto la incertidumbre de la recta como la variabilidad natural alrededor de ella.


In [ ]:
# Comparacion entre intervalos de confianza y de prediccion.
new_points <- data.frame(speed = c(12, 20, 24))
conf_int <- predict(model, newdata = new_points, interval = 'confidence')
pred_int <- predict(model, newdata = new_points, interval = 'prediction')

data.frame(
  speed = new_points$speed,
  conf_fit = round(conf_int[, 'fit'], 3),
  conf_lwr = round(conf_int[, 'lwr'], 3),
  conf_upr = round(conf_int[, 'upr'], 3),
  pred_fit = round(pred_int[, 'fit'], 3),
  pred_lwr = round(pred_int[, 'lwr'], 3),
  pred_upr = round(pred_int[, 'upr'], 3)
)


## 10) Supuestos y diagnosticos basicos

La regresion lineal simple descansa en ideas como estas:

1. la relacion promedio entre `x` y `y` es aproximadamente lineal,
2. los residuos no muestran patrones sistematicos fuertes,
3. la variabilidad de los residuos no cambia de forma extrema,
4. los errores no tienen dependencias raras entre si,
5. para cierta inferencia clasica, es util que los errores sean aproximadamente normales.

No hace falta dominar todo esto de golpe.
Pero si conviene saber que un modelo lineal no debe aceptarse solo porque produjo una recta.


In [ ]:
# Graficas diagnosticas basicas de lm().
par(mfrow = c(2, 2))
plot(model)
par(mfrow = c(1, 1))


## 11) Cuidado con extrapolar

El modelo aprende sobre el rango de velocidades que vio en los datos.
Predecir mucho mas alla de ese rango puede ser arriesgado.

Una recta puede parecer razonable entre 4 y 25, pero no necesariamente a 35 o 50.
Extrapolar no es solo "usar la formula": es suponer que la misma relacion sigue valiendo fuera del rango observado.


In [ ]:
# Predicciones dentro y fuera del rango observado.
range(cars$speed)

extra_points <- data.frame(speed = c(5, 10, 20, 25, 35))
extra_pred <- predict(model, newdata = extra_points)

data.frame(
  speed = extra_points$speed,
  dist_predicha = round(extra_pred, 3)
)


## 12) Errores comunes en regresion lineal simple

1. Confundir correlacion con causalidad.
2. Interpretar el intercepto sin pensar si `x = 0` tiene sentido.
3. Quedarse solo con `R^2` y olvidar residuos y contexto.
4. Usar el modelo para extrapolar sin cuidado.
5. Suponer que una relacion curvada se arregla por fuerza con una recta.

Si puedes explicar por que cada uno de estos errores es peligroso, ya estas entendiendo mas que solo la sintaxis de `lm()`.


In [ ]:
# Checklist minimo de ideas clave.
checklist <- c(
  'Puedo explicar que representa una pendiente en una regresion lineal simple.',
  'Puedo distinguir entre valor observado, valor predicho y residuo.',
  'Puedo interpretar R2 como proporcion de variabilidad explicada.',
  'Puedo explicar por que un intervalo de prediccion es mas ancho que uno de confianza.',
  'Puedo argumentar por que extrapolar puede ser riesgoso.'
)

checklist


## 13) Ejercicios para pensar (no copiar codigo)

Primero responde en lenguaje natural. Luego, si hace falta, usa R para apoyar tu argumento.

1. Si la pendiente de un modelo es positiva, explica que se puede concluir y que no se puede concluir.
2. En este dataset, que podria significar fisicamente un residuo muy grande y positivo.
3. Imagina que `R^2` fuera bajo pero la pendiente siguiera siendo claramente positiva. Explica por que eso no es una contradiccion.
4. Dise?a una situacion donde una regresion lineal simple sea una mala idea aunque la correlacion no sea cero.
5. Compara conceptualmente un intervalo de confianza y un intervalo de prediccion para `speed = 20`. Cual deberia ser mas ancho y por que.
6. Propone una razon por la que una variable omitida podria distorsionar la interpretacion de la relacion entre velocidad y distancia.
7. Si tuvieras que convencer a alguien de no extrapolar a `speed = 60`, que argumento tecnico y que argumento intuitivo usarias.


In [ ]:
# Espacio para resolver ejercicios.
# Primero argumenta con palabras.
# Luego valida con calculos, graficas o experimentos si hace falta.
